# Assignment 5 - Titanic Survival using Naive Bayes

In this assignment I will try to predict if passengers survived the Titanic using the Naive Bayes algorithm. 
I'm using the original Titanic-Dataset.csv file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

# first let's load the data to see what it looks like
df = pd.read_csv('Titanic-Dataset.csv')
df.head()

## Exploring and Cleaning the Data
Let's check if there are any missing values because I know machine learning models hate missing data.

In [ ]:
print(df.isnull().sum())

Wow, Cabin is missing 687 values! That's too much, I think I'll just delete that column entirely. Age is missing 177, maybe I can just fill those with the average age so I don't lose that data.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())

# Embarked only has 2 missing, I'll just fill it with 'S' since most people boarded there
df['Embarked'] = df['Embarked'].fillna('S')

# getting rid of columns that I don't think will help predict survival (like their name or ticket number)
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

print("Checking nulls again:")
print(df.isnull().sum())

## Dealing with Text (Strings)
I need to convert the text columns to numbers. 

In [ ]:
# I tried to just train the model right away but it crashed because of the text columns!
# Here was my first attempt that gave me a ValueError:
# model = GaussianNB()
# model.fit(df.drop('Survived', axis=1), df['Survived'])
# ERROR: ValueError: could not convert string to float: 'male'

# So now I'm fixing it by turning 'male' and 'female' into 0 and 1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# for the embarked column, I'll use pandas get_dummies to turn the letters into columns of 0s and 1s
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

df.head()

## Splitting the Data
Now the data looks ready. I'll split it into training data (to teach the model) and testing data (to grade it).

In [ ]:
X = df.drop('Survived', axis=1)
y = df['Survived']

# taking 20% of the data for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training data size:", len(X_train))
print("Testing data size:", len(X_test))

## Training the Naive Bayes Model
The assignment asks us to use Naive Bayes. I'll use GaussianNB since we have continuous numbers like Age and Fare.

In [ ]:
nb_model = GaussianNB()

# teaching the model
nb_model.fit(X_train, y_train)
print("Model trained successfully!")

## Results and Evaluation
Let's see how well it learned.

In [ ]:
predictions = nb_model.predict(X_test)

acc = accuracy_score(y_test, predictions)
print("Accuracy Score:", acc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

### Conclusion
The Naive Bayes model scored about 77% accuracy on the testing data. 
It's pretty cool how fast it trains compared to what I expected. 
If I had more time, maybe I could try to figure out what to do with the `Cabin` column instead of just deleting it, because maybe people in certain cabins survived more? But for now, dropping it was the easiest way to get the model working since it had so many missing values. The hardest part of this assignment was definitely figuring out that I needed to convert the text to numbers first!